In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.text_cell_render.rendered_html{font size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input{font-family:Consolas; font-size:12pt;}
div.prompt {min width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe {font-size:12px;}
</style>
"""))

# 벡터DB : Chroma vs Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB
- Pinecone : 클라우드 vector DB
    - (https//www.pinecone.io에서 api key 생성 -> .env 추가 (PINECONE_API_KEY 등록)
   

# 0. 패키지 설치

In [2]:
%pip install -q pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# 1. knowledge Base 구성을 위한 데이터 생성

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

document_list = loader.load_and_split(text_splitter=text_splitter)
len(document_list)

193

In [3]:
#embedding : upstage의 solar-embedding-1-large-passage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(model='solar-embedding-1-large-passage')

In [5]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os

pc = Pinecone(
    api_key=os.getenv('PINECONE_API_KEY')
)

# 데이터 업로드 할 때
# index_name = 'tax-index-upstage'
# database = PineconeVectorStore.from_documents(
#     documents=document_list,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드시 경고가 안보이려면 아나콘다 프롬프트 llm 환경에서 conda install -c conda forge ipywidgets

CPU times: total: 7.05 s
Wall time: 43.5 s


In [ ]:
#업로드한 벡터 db를 가져올 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

# 2. 답변 생성을 위한 Retrieval

In [6]:
query = '연봉이 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=3)

In [7]:
# retrieved_docs[2].page_content
retrieved_doc = '\n\n---\n\n'.join([doc.page_content for doc in retrieved_docs])
print(retrieved_doc)

2. 2명인 경우: 연 55만원

3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액

② 삭제<2017. 12. 19.>

③ 해당 과세기간에 출산하거나 입양 신고한 공제대상자녀가 있는 경우 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다.<신설 2015. 5. 13., 2016. 12. 20.>

1. 출산하거나 입양 신고한 공제대상자녀가 첫째인 경우: 연 30만원

2. 출산하거나 입양 신고한 공제대상자녀가 둘째인 경우: 연 50만원

3. 출산하거나 입양 신고한 공제대상자녀가 셋째 이상인 경우: 연 70만원

④ 제1항 및 제3항에 따른 공제를 “자녀세액공제”라 한다.<신설 2015. 5. 13., 2017. 12. 19.>

[본조신설 2014. 1. 1.]

[종전 제59조의2는 제59조의5로 이동 <2014. 1. 1.>]



제59조의3(연금계좌세액공제) ① 종합소득이 있는 거주자가 연금계좌에 납입한 금액 중 다음 각 호에 해당하는 금액을 제외한 금액(이하 “연금계좌 납입액”이라 한다)의 100분의 12[해당 과세기간에 종합소득과세표준을 계산할 때 합산하는 종합소득금액이 4천 500만원 이하(근로소득만 있는 경우에는 총급여액 5천 500만원 이하)인 거주자에 대해서는 100분의 15]에 해당하는 금액을 해당 과세기간의 종합소득산출세액에서 공제한다. 다만, 연금계좌 중 연금저축계좌에 납입한 금액이 연 600만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 하고, 연금저축계좌에 납입한 금액 중 600만원 이내의 금액과 퇴직연금계좌에 납입한 금액을 합한 금액이 연 900만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 한다. <개정 2014. 12. 23., 2015. 5. 13., 2016. 12. 20., 2022. 12. 31.>

1. 제146조제2항에 따라 소득세가 원천징수되지 아니한 퇴직소득 등 과세가 이연된 소득

2. 연금계좌에서 다른 연금계좌로 계약을 이전함으로써 납입되

In [7]:
retriever = database.as_retriever(
    search_kwargs={'k':3}
)
retrieved_docs = retriever.invoke(query)

In [8]:
# retrieved_docs[2].page_content
retrieved_doc = '\n\n---\n\n'.join([doc.page_content for doc in retrieved_docs])
print(retrieved_doc)

[전문개정 2009. 12. 31.]



제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다.

[전문개정 2009. 12. 31.]



제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다.

[전문개정 2009. 12. 31.]



제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>



제1절 비과세 <개정 2009. 12. 31.>



제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.>

1. 「공익신탁법」에 따른 공익신탁의 이익

2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득

가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득

나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지

# 3. 답변 생성

In [9]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [8]:
# upstatge에서 받은 $20로 llm을 사용하고 싶다면
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model='solar-pro2',
    reasoning_effort='high' #느리지만 더 깊게 추론함 (low, medium)
)

In [9]:
prompt=f'''[identity]
- 당신은 최고의 한국 소득세법 전문가입니다
- [context]를 참고해서 사용자의 질문에 답변해 주세요.
- [context]는 다음과 같습니다
{retrieved_doc}
- 질문:{query}'''

In [10]:
ai_message = llm.invoke(prompt)

In [11]:
ai_message.content

'연봉 5천만 원인 직장인의 소득세를 계산하기 위해 다음과 같은 단계를 거칩니다. 단, **제공된 [context]에는 근로소득 공제율 및 세율 표가 누락되어 있으므로, 일반적인 한국 소득세 계산 방식을 기준으로 설명**드립니다. 실제 계산 시에는 정확한 공제율 및 세율 확인이 필요합니다.\n\n---\n\n### **1. 총급여액: 5,000만 원**\n\n---\n\n### **2. 근로소득공제 적용**  \n근로소득공제는 총급여액에 따라 차등 적용됩니다 (2024년 기준).  \n- **5천만 원 소득자의 공제액**:  \n  - 1,500만 원 이하: 70% (1,050만 원)  \n  - 1,500만 원 초과 ~ 4,500만 원: 15% (4,500만 원 - 1,500만 원 = 3,000만 원 × 15% = 450만 원)  \n  - 4,500만 원 초과 ~ 5,000만 원: 5% (500만 원 × 5% = 25만 원)  \n  - **총 공제액**: 1,050만 + 450만 + 25만 = **1,525만 원**  \n  - **공제 후 과세표준**: 5,000만 원 - 1,525만 원 = **3,475만 원**\n\n---\n\n### **3. 종합소득 산출세액 계산**  \n과세표준에 **누진세율**을 적용합니다 (2024년 기준):  \n- 1,200만 원 이하: 6% → 1,200만 × 6% = 72만 원  \n- 1,200만 원 초과 ~ 3,475만 원: 15% (누진공제 108만 원 적용)  \n  - (3,475만 - 1,200만) × 15% = 345만 원  \n  - 345만 원 - 108만 원 = **237만 원**  \n- **산출세액 합계**: 72만 + 237만 = **309만 원**\n\n---\n\n### **4. 세액공제 적용**  \n#### **가. 근로소득세액공제** (제59조)  \n- 총급여액 5,000만 원 (3,300만 원 초과 ~ 7,000만 원 이하)  \n  - 공제액 = 74만 원 - [(5,000만 -

# 4. langchain 전달

In [3]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 1. LLM과 임베딩 초기화
load_dotenv()
llm = ChatUpstage(model='solar-pro2-251215')
embedding = UpstageEmbeddings(model='solar-embedding-1-large-passage')
# 2. vector store load
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name='tax-index-upstage'
)
# 3. Retriever 생성
retriever = database.as_retriever(
    search_kwargs={'k':3}
)
# 4. 프롬프트 템플릿
template=f'''[identity]
- 당신은 최고의 한국 소득세법 전문가입니다
- [context]를 참고해서 사용자의 질문에 답변해 주세요.
- [context]는 다음과 같습니다
{{context}}
- 질문:{{query}}'''
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 document를 텍스트로 변환하는 함수
def format_documents(documents):
    return "\n\n---\n\n".join([retrieved_doc.page_content for retrieved_doc in documents])

In [4]:
# 6. RAG 체인 구성 (LCEL 방식)
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {
        'context':retriever | format_documents,
        'query':RunnablePassthrough() # 질문 그대로 전달
    }
    | prompt # prompt에 context와 query 주입
    | llm # llm에 prompt 주입
    | StrOutputParser() # StrOutputParser에 llm 주입
)

# 7. 실행
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
rag_chain.invoke(query)

'2024년 기준 연봉 5,000만원 직장인의 소득세를 계산하는 절차는 다음과 같습니다. (※ 단순화된 계산이며, 실제 세액과는 차이가 있을 수 있습니다.)\n\n### 1. **근로소득공제 적용**\n- 총급여액 5,000만원 × 15% + 180만원 = **930만원**  \n  (※ 5,000만원 이하 구간: 70% + 1,400만원 × (총급여액 ÷ 1,200만원) 방식 적용 시 930만원)\n\n### 2. **근로소득금액 계산**\n- 총급여액 5,000만원 - 근로소득공제 930만원 = **4,070만원** (과세표준)\n\n### 3. **종합소득 산출세액 계산**\n- 4,070만원 × 기본세율(15%) - 108만원(누진공제) = **496.5만원**  \n  (※ 4,600만원 이하 구간: 15% - 108만원)\n\n### 4. **세액공제 적용**\n#### (1) 자녀세액공제 (예시: 자녀 2명)\n- 기본공제: 55만원 (2명 기준)  \n- 출산/입양 추가공제: 없음 (해당 사항 없음)\n\n#### (2) 근로소득세액공제\n- 총급여액 5,000만원 → 공제액 **66만원**  \n  (3,300만원 초과 7,000만원 이하 구간: 74만원 - (5,000-3,300)×0.8% = 66.8만원 → **66만원**)\n\n#### (3) 연금계좌세액공제 (예시: 연금저축 600만원 납입)\n- 600만원 × 15% = **90만원**  \n  (종합소득 4,500만원 이하 또는 근로소득 5,500만원 이하 시 15% 적용)\n\n### 5. **최종 결정세액**\n- 산출세액 496.5만원 - 자녀공제 55만원 - 근로소득 공제 66만원 - 연금공제 90만원 = **285.5만원**\n\n### 6. **월별 원천징수세액 (예상)**\n- 연간 세액 285.5만원 ÷ 12개월 = **월 약 23.8만원**  \n  (실제로는 중간정산 또는 연말정산 시 조정 가능)\n\n> ※ 주의: 실제 세금 계산 시에는 **의료비·교육비·신용카드